## Краткое описание
Я использовал ансамбль из CatBoost, двух моделей LightGBM и XGBoost. Помимо исходных табличных данных, я добавил агрегаты пользовательских событий до момента назначения лида: количество и давность действий, активность в разных временных окнах, долю контактных действий и разнообразие контекстов. Также я построил признаки позиции объявления в выдаче, сочетания типа события с контекстом и близости цены текущего объявления к ранее просмотренным ценам.
Для валидации я использовал временное разбиение: обучал модели на ранних датах, а качество рассчитывал отдельно на каждом из последних четырёх дней. Итоговой метрикой было среднее значение Average Precision по дням.
Я пробовал дополнительные отношения и тренды активности, последовательности событий и признаки ускорения интереса. Нестабильные признаки и признаки, приводившие к переобучению, исключил по результатам временной валидации. Финальные прогнозы моделей объединил взвешенным усреднением.

In [28]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score


# Конфигурация

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ROOT = Path(".")
DATA_DIR = ROOT / "data"
TARGET = "target"

ID_COLUMNS = {"lead_id", "user_id"}
TIME_COLUMNS = {"assignment_ts", "assignment_date"}
NON_FEATURE_COLUMNS = ID_COLUMNS | TIME_COLUMNS | {TARGET}

CATEGORICAL_FEATURES = [
    "lead_source", "call_center", "region",
    "car_segment", "lead_channel", "user_tenure_bucket", "price_bucket",
    "recency_bucket"
]

EVENT_WINDOWS_HOURS = [24, 72, 168]
EVENT_TYPES = ["item_view", "search", "favorite", "chat_open", "call_click"]
CTX_SEQ_VALUES = ["c01", "c02", "c04", "c06", "c08"]


# Гиперпараметры, подобранные на последних четырех днях

CATBOOST_PARAMS = {
    "iterations": 1783,
    "learning_rate": 0.06647,
    "depth": 4,
    "l2_leaf_reg": 26.277,
    "min_data_in_leaf": 31,
    "random_strength": 3.431,
    "bagging_temperature": 0.540,
    "random_seed": RANDOM_STATE,
    "verbose": 0,
    "eval_metric": "AUC",
    "auto_class_weights": "Balanced",
}

# Первый LightGBM использует только базовые числовые признаки. Он оставлен
# близким к исходной модели: новый поиск не дал для него честного прироста.
LGBM_BASE_PARAMS = {
    "n_estimators": 588,
    "learning_rate": 0.13106,
    "max_depth": 5,
    "num_leaves": 60,
    "min_child_samples": 26,
    "min_child_weight": 0.001,
    "reg_alpha": 7.765,
    "reg_lambda": 9.132,
    "subsample": 0.750,
    "subsample_freq": 0,
    "colsample_bytree": 0.810,
    "min_split_gain": 0.0,
    "max_bin": 255,
    "random_state": RANDOM_STATE,
    "verbose": -1,
    "n_jobs": -1,
}

# Второй LightGBM получает дополнительные 70 признаков. Для него параметры
# были заново подобраны после создания новых признаков.
LGBM_ENHANCED_PARAMS = {
    "n_estimators": 831,
    "learning_rate": 0.02848899077090316,
    "max_depth": 6,
    "num_leaves": 93,
    "min_child_samples": 74,
    "min_child_weight": 0.1971377303522683,
    "reg_alpha": 2.9130615452989885,
    "reg_lambda": 1.6777683317954246,
    "subsample": 0.9141046087089012,
    "subsample_freq": 1,
    "colsample_bytree": 0.6823185692092332,
    "min_split_gain": 0.16008985451735502,
    "max_bin": 255,
    "random_state": RANDOM_STATE,
    "verbose": -1,
    "n_jobs": -1,
    "metric": "None",
}

XGBOOST_PARAMS = {
    "n_estimators": 2179,
    "learning_rate": 0.03291160846030842,
    "max_depth": 4,
    "min_child_weight": 5.17128606798581,
    "reg_alpha": 0.010234263483376888,
    "reg_lambda": 0.0011184218258643121,
    "subsample": 0.9096089299975761,
    "colsample_bytree": 0.8286760989354791,
    "gamma": 0.3486059462764247,
    "random_state": RANDOM_STATE,
    "verbosity": 0,
    "n_jobs": -1,
    "eval_metric": "logloss",
}

# Порядок: CatBoost, базовый LightGBM, расширенный LightGBM, XGBoost.
ENSEMBLE_WEIGHTS = np.array([0.250, 0.225, 0.275, 0.250])


# Признаки для удаления по результатам исследований

FEATURES_TO_DROP = {
    "rel_search_views_7d_to_90d", "similar_item_clicks_30d",
    "seller_response_rate_30d", "photo_swipes_30d",
    "trend_photo_swipes_30d_vs_14d", "trend_call_clicks_30d_vs_14d",
    "rel_call_clicks_7d_to_90d", "rel_chat_opens_7d_to_90d",
    "item_price_log", "chat_opens_90d", "evt_mean_price",
    "trend_item_favorites_7d_vs_3d",
    "ratio_positive_assigned_3d", "leadgen_prev_assigned_90d",
    "rel_user_contacts_7d_to_30d", "ratio_positive_assigned_7d",
    "trend_search_views_14d_vs_7d", "active_days_auto_14d",
    "rel_call_clicks_7d_to_30d", "evt_span_hours", "trend_chat_opens_14d_vs_7d",
    "seller_page_views_3d", "assignment_weekday", "similar_item_clicks_3d",
    "saved_search_matches_7d", "active_days_auto_3d",
    "item_favorites_14d", "search_views_3d", "item_favorites_3d",
    "total_contact_actions_7d", "leadgen_prev_assigned_7d",
    "active_days_auto_7d", "call_clicks_90d", "user_contacts_14d",
    "ratio_answered_assigned_14d", "rel_item_views_7d_to_90d",
    "trend_item_views_14d_vs_7d", "trend_item_views_90d_vs_30d",
    "leadgen_prev_positive_14d", "ratio_positive_answered_90d",
    "ratio_answered_assigned_1d", "ratio_positive_assigned_1d",
    "ratio_positive_answered_1d", "ratio_positive_answered_7d",
    "interact_price_x_age", "ratio_answered_assigned_3d",
    "ratio_positive_assigned_14d", "ratio_positive_answered_14d",
    "lead_channel", "is_weekend",
    "query_refinements_14d", "trend_detail_expands_30d_vs_14d",
    "trend_active_days_auto_14d_vs_7d", "leadgen_prev_assigned_3d",
    "chat_opens_14d", "total_activity_7d", "detail_expands_1d",
    "evt_item_view_168h", "user_contacts_1d", "leadgen_prev_positive_1d",
    "query_refinements_3d", "query_refinements_30d",
    "leadgen_prev_positive_90d",
}


##1. Загрузка данных

In [37]:
def load_data():
    train = pd.read_csv(DATA_DIR / "train.csv")
    test = pd.read_csv(DATA_DIR / "test.csv")
    events = pd.read_csv(DATA_DIR / "events.csv")
    train["assignment_ts"] = pd.to_datetime(train["assignment_ts"])
    test["assignment_ts"] = pd.to_datetime(test["assignment_ts"])
    events["event_ts"] = pd.to_datetime(events["event_ts"])
    return train, test, events

##2. Базовые признаки из истории событий

In [30]:
def build_event_features(df: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:
    merged = df[["lead_id", "assignment_ts"]].merge(events, on="lead_id", how="left")
    merged = merged[merged["event_ts"] < merged["assignment_ts"]].copy()
    merged["hours_before"] = (
        (merged["assignment_ts"] - merged["event_ts"]).dt.total_seconds() / 3600
    )

    result = pd.DataFrame({"lead_id": df["lead_id"].values})

    # Счетчики по временным окнам и типам событий.
    for wh in EVENT_WINDOWS_HOURS:
        wn = f"{wh}h"
        we = merged[merged["hours_before"] <= wh]
        total = we.groupby("lead_id").size().rename(f"evt_total_{wn}")
        result = result.merge(total, on="lead_id", how="left")
        for et in EVENT_TYPES:
            c = we[we["event_type"] == et].groupby("lead_id").size().rename(f"evt_{et}_{wn}")
            result = result.merge(c, on="lead_id", how="left")

    # Общие агрегаты
    agg = merged.groupby("lead_id").agg(
        evt_total_all=("event_ts", "size"),
        evt_n_unique_types=("event_type", "nunique"),
        evt_n_unique_ctx=("ctx_seq", "nunique"),
        evt_mean_src_slot=("src_slot", "mean"),
        evt_std_src_slot=("src_slot", "std"),
        evt_recency_hours=("hours_before", "min"),
    )
    result = result.merge(agg, on="lead_id", how="left")

    # Распределение событий по значениям контекста.
    for cv in CTX_SEQ_VALUES:
        c = merged[merged["ctx_seq"] == cv].groupby("lead_id").size().rename(f"evt_ctx_{cv}_count")
        result = result.merge(c, on="lead_id", how="left")
    for cv in CTX_SEQ_VALUES:
        result[f"evt_ctx_{cv}_share"] = (
            result[f"evt_ctx_{cv}_count"] / result["evt_total_all"].replace(0, np.nan)
        )

    # Энтропия отражает разнообразие пользовательских контекстов.
    ctx_cols = [f"evt_ctx_{v}_count" for v in CTX_SEQ_VALUES]
    ctx_df = result[ctx_cols].fillna(0)
    ctx_total = ctx_df.sum(axis=1).replace(0, np.nan)
    ctx_probs = ctx_df.div(ctx_total, axis=0)
    result["evt_ctx_entropy"] = -(ctx_probs * np.log(ctx_probs + 1e-10)).sum(axis=1)

    # Отношение добавлений в избранное к просмотрам.
    for wh in EVENT_WINDOWS_HOURS:
        wn = f"{wh}h"
        fav, view = f"evt_favorite_{wn}", f"evt_item_view_{wn}"
        if fav in result.columns and view in result.columns:
            result[f"evt_fav_per_view_{wn}"] = result[fav] / result[view].replace(0, np.nan)

    # Доля контактных действий среди всех событий.
    for wh in EVENT_WINDOWS_HOURS:
        wn = f"{wh}h"
        ch, ca, tot = f"evt_chat_open_{wn}", f"evt_call_click_{wn}", f"evt_total_{wn}"
        if all(c in result.columns for c in [ch, ca, tot]):
            result[f"evt_contact_share_{wn}"] = (
                (result[ch].fillna(0) + result[ca].fillna(0)) / result[tot].replace(0, np.nan)
            )

    # Число активных дней и средняя интенсивность.
    ad = (merged.assign(ed=merged["event_ts"].dt.date)
          .groupby("lead_id")["ed"].nunique().rename("evt_active_days"))
    result = result.merge(ad, on="lead_id", how="left")
    result["evt_intensity"] = result["evt_total_all"] / result["evt_active_days"].replace(0, np.nan)

    # Доли типов событий за семь дней.
    for et in EVENT_TYPES:
        col = f"evt_{et}_168h"
        if col in result.columns:
            result[f"evt_{et}_share_7d"] = result[col] / result["evt_total_168h"].replace(0, np.nan)

    # Разнообразие контекстов в коротких временных окнах.
    for hours, name in [(72, "72h"), (24, "24h")]:
        w = merged[merged["hours_before"] <= hours]
        nu = w.groupby("lead_id")["ctx_seq"].nunique().rename(f"evt_n_unique_ctx_{name}")
        result = result.merge(nu, on="lead_id", how="left")

    return result

##3. Проверенные табличные сегменты и взаимодействия

In [31]:
def build_tabular_features(df: pd.DataFrame) -> pd.DataFrame:

    # Оставлены только сегменты, показавшие стабильный прирост при отключениях.

    result = df.copy()

    # Индикаторы источника лида.
    result["is_crm"] = (result["lead_source"] == "CRM").astype(int)
    result["is_model"] = (result["lead_source"] == "Model").astype(int)
    result["is_perf"] = (result["lead_source"] == "Perf").astype(int)

    # Низкая активность: меньше медианного числа событий, примерно равного 12.
    evt_total = result.get("evt_total_all", pd.Series(0, index=result.index)).fillna(0)
    result["is_low_activity"] = (evt_total < 12).astype(int)

    # Категориальный интервал давности последнего события.
    recency = result.get("evt_recency_hours", pd.Series(np.nan, index=result.index))
    result["recency_bucket"] = pd.cut(
        recency, bins=[-1, 6, 24, 72, float("inf")],
        labels=["0-6h", "6-24h", "24-72h", "72h+"],
    ).astype(str).replace("nan", "no_events")

    result["has_recent_event_24h"] = (recency <= 24).astype(int).fillna(0)
    result["has_recent_event_72h"] = (recency <= 72).astype(int).fillna(0)
    result["has_seller_views"] = (result.get("seller_page_views_14d", 0) > 0).astype(int)

    spv7 = result.get("seller_page_views_7d", pd.Series(0))
    spv14 = result.get("seller_page_views_14d", pd.Series(0))
    result["seller_views_trend"] = spv7 / (spv14 + 1)

    # Взаимодействия источника CRM с пользовательской активностью.
    result["crm_x_low_activity"] = result["is_crm"] * result["is_low_activity"]
    result["crm_x_recency"] = result["is_crm"] * recency.fillna(999)
    result["crm_x_seller_views"] = result["is_crm"] * result.get("seller_page_views_14d", 0)
    result["crm_x_has_recent_24h"] = result["is_crm"] * result["has_recent_event_24h"]

    result["low_act_x_seller_views"] = result["is_low_activity"] * result.get("seller_page_views_14d", 0)
    result["low_act_x_item_views"] = result["is_low_activity"] * result.get("item_views_90d", 0)
    result["low_act_x_search_views"] = result["is_low_activity"] * result.get("search_views_90d", 0)
    result["low_act_x_seller_inv"] = result["is_low_activity"] * result.get("seller_inventory_count", 0)

    # Отношения исторических показателей за 30 и 90 дней.
    for window in ["30d", "90d"]:
        assigned = f"leadgen_prev_assigned_{window}"
        answered = f"leadgen_prev_answered_{window}"
        positive = f"leadgen_prev_positive_{window}"
        if assigned in result.columns and answered in result.columns:
            result[f"ratio_answered_assigned_{window}"] = (
                result[answered] / result[assigned].replace(0, np.nan)
            )
        if assigned in result.columns and positive in result.columns:
            result[f"ratio_positive_assigned_{window}"] = (
                result[positive] / result[assigned].replace(0, np.nan)
            )

    # Изменения активности между длинным и коротким окнами.
    trend_pfx = [
        "item_views", "item_favorites", "detail_expands", "photo_swipes",
        "seller_page_views", "search_views", "user_contacts", "chat_opens",
        "call_clicks", "active_days_auto",
    ]
    for prefix in trend_pfx:
        for wl, ws in [("30d", "14d"), ("90d", "30d")]:
            cl, cs = f"{prefix}_{wl}", f"{prefix}_{ws}"
            if cl in result.columns and cs in result.columns:
                result[f"trend_{prefix}_{wl}_vs_{ws}"] = result[cl] - result[cs]

    # Относительная краткосрочная активность за 3 и 30 дней.
    for prefix in ["item_views", "search_views", "user_contacts", "chat_opens", "call_clicks"]:
        cs, cl = f"{prefix}_3d", f"{prefix}_30d"
        if cs in result.columns and cl in result.columns:
            result[f"rel_{prefix}_3d_to_30d"] = result[cs] / result[cl].replace(0, np.nan)

    # Базовые взаимодействия числовых признаков.
    if "user_active_days_30d" in result.columns and "prior_assignments_30d" in result.columns:
        result["interact_activity_x_assignments"] = (
            result["user_active_days_30d"] * result["prior_assignments_30d"]
        )
    if "seller_inventory_count" in result.columns and "seller_response_rate_30d" in result.columns:
        result["interact_seller_inv_x_response"] = (
            result["seller_inventory_count"] * result["seller_response_rate_30d"]
        )

    return result

## 4. Дополнительные признаки, прошедшие локальную проверку

In [32]:
def build_src_slot_features(df: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:

    #Агрегаты позиции источника в истории событий пользователя.
    leads = df[["lead_id", "assignment_ts"]]
    merged = leads.merge(
        events[["lead_id", "event_ts", "src_slot"]], on="lead_id", how="left",
    )
    merged = merged[merged["event_ts"] < merged["assignment_ts"]].copy()
    merged["hours_before"] = (
        merged["assignment_ts"] - merged["event_ts"]
    ).dt.total_seconds() / 3600
    valid = merged.dropna(subset=["src_slot"])

    result = valid.groupby("lead_id")["src_slot"].agg(
        slot_median="median",
        slot_min="min",
        slot_max="max",
    )
    last = (
        valid.sort_values("event_ts").groupby("lead_id").tail(1)
        .set_index("lead_id")["src_slot"].rename("slot_last")
    )
    result = result.join(last, how="outer")

    # Доля взаимодействий с объектами в верхних позициях выдачи.
    for cutoff in [3, 5, 10]:
        result[f"slot_top{cutoff}_share"] = (
            (valid["src_slot"] <= cutoff).groupby(valid["lead_id"]).mean()
        )
    result["slot_mean_reciprocal"] = (
        (1.0 / valid["src_slot"].clip(lower=1)).groupby(valid["lead_id"]).mean()
    )

    # Последние окна позволяют отличить свежий интерес от старой истории.
    for hours in [24, 72, 168]:
        result[f"slot_mean_{hours}h"] = (
            valid[valid["hours_before"] <= hours]
            .groupby("lead_id")["src_slot"].mean()
        )
    result["slot_recent_vs_all"] = result["slot_mean_72h"] - result["slot_median"]
    return result.reset_index()


def build_ctx_event_crosses(df: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:

    #Счетчики типов контактных действий внутри каждого контекста.
    leads = df[["lead_id", "assignment_ts"]]
    merged = leads.merge(
        events[["lead_id", "event_ts", "event_type", "ctx_seq"]],
        on="lead_id",
        how="left",
    )
    merged = merged[merged["event_ts"] < merged["assignment_ts"]].copy()
    merged["ctx_seq"] = merged["ctx_seq"].fillna("missing")
    merged["is_contact"] = merged["event_type"].isin(
        ["favorite", "chat_open", "call_click"],
    ).astype(int)
    merged["is_chat"] = (merged["event_type"] == "chat_open").astype(int)
    merged["is_call"] = (merged["event_type"] == "call_click").astype(int)

    grouped = merged.groupby(["lead_id", "ctx_seq"]).agg(
        total=("event_type", "size"),
        contact=("is_contact", "sum"),
        chat=("is_chat", "sum"),
        call=("is_call", "sum"),
    )
    wide = grouped.unstack(fill_value=0)
    wide.columns = [f"ctxevent_{metric}_{ctx}" for metric, ctx in wide.columns]
    result = wide.reset_index()

    for ctx in sorted(merged["ctx_seq"].unique()):
        total = f"ctxevent_total_{ctx}"
        contact = f"ctxevent_contact_{ctx}"
        if total in result.columns and contact in result.columns:
            result[f"ctxevent_contact_share_{ctx}"] = (
                result[contact] / (result[total] + 1.0)
            )
    return result


def build_price_affinity(df: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:

    #Сравнение цены текущего объявления с историей просмотренных цен.
    leads = df[["lead_id", "assignment_ts", "item_price_log"]].rename(
        columns={"item_price_log": "current_price"},
    )
    event_data = events[["lead_id", "event_ts", "item_price_log"]].rename(
        columns={"item_price_log": "event_price"},
    )
    merged = leads.merge(event_data, on="lead_id", how="left")
    merged = merged[merged["event_ts"] < merged["assignment_ts"]].copy()
    merged["hours_before"] = (
        merged["assignment_ts"] - merged["event_ts"]
    ).dt.total_seconds() / 3600
    priced = merged.dropna(subset=["event_price"]).copy()

    aggregates = priced.groupby("lead_id")["event_price"].agg(
        price_hist_mean="mean",
        price_hist_median="median",
        price_hist_std="std",
        price_hist_min="min",
        price_hist_max="max",
        price_hist_count="count",
    )
    last = (
        priced.sort_values("event_ts").groupby("lead_id").tail(1)
        .set_index("lead_id")["event_price"].rename("price_hist_last")
    )
    recent = (
        priced[priced["hours_before"] <= 168]
        .groupby("lead_id")["event_price"].mean().rename("price_hist_mean_7d")
    )
    priced["price_close_025"] = (
        (priced["event_price"] - priced["current_price"]).abs() <= 0.25
    ).astype(float)
    close = (
        priced.groupby("lead_id")["price_close_025"]
        .mean().rename("price_hist_close_share")
    )

    result = aggregates.join([last, recent, close], how="outer").reset_index()
    result = df[["lead_id", "item_price_log"]].merge(result, on="lead_id", how="left")
    for reference in ["mean", "median", "last", "mean_7d"]:
        history = f"price_hist_{reference}"
        gap = f"price_gap_{reference}"
        result[gap] = result["item_price_log"] - result[history]
        result[f"price_abs_gap_{reference}"] = result[gap].abs()
    result["price_gap_z"] = (
        result["price_gap_mean"] / result["price_hist_std"].clip(lower=0.05)
    )
    return result.drop(columns="item_price_log")


def add_validated_features(df: pd.DataFrame, events: pd.DataFrame):

    #Добавляет три проверенных блока и возвращает имена новых столбцов.
    result = df.copy()
    added = []
    for builder in [
        build_src_slot_features,
        build_ctx_event_crosses,
        build_price_affinity,
    ]:
        feature_block = builder(result, events)
        result = result.merge(feature_block, on="lead_id", how="left")
        added.extend(column for column in feature_block.columns if column != "lead_id")
    result[added] = result[added].replace([np.inf, -np.inf], np.nan)
    return result, added

## 5. Отбор признаков и метрика

In [33]:
def select_features(feature_cols):
    return [c for c in feature_cols if c not in FEATURES_TO_DROP]


def daily_average_precision(y_true, y_score, dates):
    daily_aps = []
    for date in np.unique(dates):
        mask = dates == date
        if y_true[mask].sum() == 0:
            continue
        daily_aps.append(average_precision_score(y_true[mask], y_score[mask]))
    return np.mean(daily_aps) if daily_aps else 0.0


def make_time_split(df, val_days=4):
    dates = pd.to_datetime(df["assignment_date"]).dt.date
    ordered = sorted(dates.unique())
    cutoff = ordered[-val_days]
    tr = df[dates < cutoff].copy()
    vl = df[dates >= cutoff].copy()
    return tr, vl


def get_feature_columns(df):
    return [c for c in df.columns if c not in NON_FEATURE_COLUMNS]

## 6. Обучение моделей с фиксированным числом итераций

In [34]:
def prepare_catboost(df, features, categorical):
    #CatBoost принимает категориальные пропуски только после явного заполнения.

    result = df[features].copy()
    for column in categorical:
        result[column] = result[column].fillna("missing").astype(str)
    return result


def train_catboost(X_train, y_train, categorical):
    actual_cats = [column for column in categorical if column in X_train.columns]
    for col in actual_cats:
        X_train[col] = X_train[col].fillna("missing").astype(str)
    params = CATBOOST_PARAMS.copy()
    params["cat_features"] = actual_cats
    model = CatBoostClassifier(**params)
    model.fit(X_train, y_train, verbose=False)
    return model


def train_lgbm(X_train, y_train, model_config):
    positive = int(y_train.sum())
    negative = len(y_train) - positive
    params = model_config.copy()
    params["scale_pos_weight"] = negative / max(positive, 1)
    model = LGBMClassifier(**params)
    model.fit(X_train, y_train)
    return model


def train_xgboost(X_train, y_train):
    positive = int(y_train.sum())
    negative = len(y_train) - positive
    params = XGBOOST_PARAMS.copy()
    params["scale_pos_weight"] = negative / max(positive, 1)
    model = XGBClassifier(**params)
    model.fit(X_train, y_train, verbose=False)
    return model


def blend_predictions(catboost, lgbm_base, lgbm_enhanced, xgboost):
    #Смешивает четыре модели в порядке, использованном при подборе весов.

    predictions = [catboost, lgbm_base, lgbm_enhanced, xgboost]
    return sum(weight * prediction for weight, prediction in zip(ENSEMBLE_WEIGHTS, predictions))

## 7. Основной пайплайн

In [40]:
def run_pipeline():
    # 1. Загружаем исходные таблицы и сразу преобразуем временные столбцы.
    train, test, events = load_data()

    # 2. Базовые агрегаты событий строятся строго до момента назначения лида.
    train_ev = build_event_features(train, events)
    test_ev = build_event_features(test, events)
    train = train.merge(train_ev, on="lead_id", how="left")
    test = test.merge(test_ev, on="lead_id", how="left")

    # 3. Сегменты, отношения и взаимодействия из исходной сильной модели.
    train = build_tabular_features(train)
    test = build_tabular_features(test)
    base_features = select_features(get_feature_columns(train))
    categorical = [
        column for column in CATEGORICAL_FEATURES if column in base_features
    ]
    base_numeric = [
        column for column in base_features if column not in categorical
    ]

    # 4. Три новых блока были проверены отдельно и вместе на последних днях.
    train, added_features = add_validated_features(train, events)
    test, _ = add_validated_features(test, events)

    # Если в тесте отсутствует целая категория ctx_seq, соответствующий счетчик
    # должен быть нулевым. Новые категории теста просто не входят в схему обучения.
    for column in added_features:
        if column not in test.columns:
            test[column] = 0.0 if column.startswith("ctxevent_") else np.nan

    enhanced_features = base_features + added_features
    enhanced_numeric = base_numeric + added_features
    train[enhanced_numeric] = train[enhanced_numeric].replace([np.inf, -np.inf], np.nan)
    test[enhanced_numeric] = test[enhanced_numeric].replace([np.inf, -np.inf], np.nan)

    # 5. Локальная валидация: четыре последних известных дня.
    train_part, valid_part = make_time_split(train, val_days=4)
    y_train = train_part[TARGET].to_numpy()
    y_valid = valid_part[TARGET].to_numpy()
    valid_dates = pd.to_datetime(valid_part["assignment_date"]).dt.date.to_numpy()

    # CatBoost, расширенный LightGBM и XGBoost используют все новые признаки.
    X_cb_train = prepare_catboost(train_part, enhanced_features, categorical)
    X_cb_valid = prepare_catboost(valid_part, enhanced_features, categorical)
    cb_model = train_catboost(X_cb_train, y_train, categorical)
    cb_valid = cb_model.predict_proba(X_cb_valid)[:, 1]

    lgb_base_model = train_lgbm(
        train_part[base_numeric], y_train, LGBM_BASE_PARAMS,
    )
    lgb_base_valid = lgb_base_model.predict_proba(valid_part[base_numeric])[:, 1]

    lgb_enhanced_model = train_lgbm(
        train_part[enhanced_numeric], y_train, LGBM_ENHANCED_PARAMS,
    )
    lgb_enhanced_valid = lgb_enhanced_model.predict_proba(
        valid_part[enhanced_numeric],
    )[:, 1]

    xgb_model = train_xgboost(train_part[enhanced_numeric], y_train)
    xgb_valid = xgb_model.predict_proba(valid_part[enhanced_numeric])[:, 1]

    validation_scores = blend_predictions(
        cb_valid, lgb_base_valid, lgb_enhanced_valid, xgb_valid,
    )
    validation_dap = daily_average_precision(
        y_valid, validation_scores, valid_dates,
    )
    print(f"Validation Daily AP: {validation_dap:.6f}")

    # 6. Переобучаем те же модели на всех размеченных данных.
    y_full = train[TARGET].to_numpy()

    X_cb_full = prepare_catboost(train, enhanced_features, categorical)
    X_cb_test = prepare_catboost(test, enhanced_features, categorical)
    cb_final = train_catboost(X_cb_full, y_full, categorical)
    cb_test = cb_final.predict_proba(X_cb_test)[:, 1]

    lgb_base_final = train_lgbm(train[base_numeric], y_full, LGBM_BASE_PARAMS)
    lgb_base_test = lgb_base_final.predict_proba(test[base_numeric])[:, 1]

    lgb_enhanced_final = train_lgbm(
        train[enhanced_numeric], y_full, LGBM_ENHANCED_PARAMS,
    )
    lgb_enhanced_test = lgb_enhanced_final.predict_proba(
        test[enhanced_numeric],
    )[:, 1]

    xgb_final = train_xgboost(train[enhanced_numeric], y_full)
    xgb_test = xgb_final.predict_proba(test[enhanced_numeric])[:, 1]

    test_scores = blend_predictions(
        cb_test, lgb_base_test, lgb_enhanced_test, xgb_test,
    )
    test_scores = np.clip(test_scores, 0.001, 0.999)

    # 7. Формируем отдельный финальный файл, не затирая старые отправки.
    submission = pd.DataFrame({
        "lead_id": test["lead_id"].astype(str),
        "score": test_scores,
    })
    submission.to_csv(ROOT / "submission.csv", index=False)

    assert list(submission.columns) == ["lead_id", "score"]
    assert len(submission) == len(test)
    assert submission["lead_id"].is_unique
    assert np.isfinite(submission["score"]).all()
    assert submission["score"].between(0, 1).all()

    print(f"submission.csv готов: {len(submission)} строк")

run_pipeline()

Validation Daily AP: 0.745937
submission.csv готов: 4306 строк
